# METplus to Parquet

Demonstrates converting METplus GridStat `.stat` output files to Parquet format and exploring the result.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq

from convert_functions import convert

## Settings

Define human-readable column labels and mark the most useful columns.  
Set `KEY_COLUMNS_ONLY = True` to drop non-key columns from the dataset.

In [ ]:
COLUMN_INFO = {
    # col name          label                            key column?
    "INIT_DATE":       ("Initialisation Date",           True),
    "FCST_LEAD_H":     ("Forecast Lead Time (hours)",    True),
    "FCST_VALID_BEG":  ("Valid Time",                    True),
    "FCST_LEV":        ("Pressure Level",                True),
    "VX_MASK":         ("Domain / Region",               True),
    "TOTAL":           ("Number of Observations",        True),
    "ME":              ("Mean Error (Bias)",              True),
    "RMSE":            ("Root Mean Square Error",        True),
    "MAE":             ("Mean Absolute Error",           True),
    "FBAR":            ("Mean Forecast Value",           False),
    "OBAR":            ("Mean Observed Value",           False),
    "FCST_VALID_END":  ("Valid Time End",                False),
    "OBS_VALID_BEG":   ("Obs Valid Time Start",          False),
    "OBS_VALID_END":   ("Obs Valid Time End",            False),
    "FOBAR":           ("Mean Forecast * Obs",           False),
    "FFBAR":           ("Mean Forecast Squared",         False),
    "OOBAR":           ("Mean Obs Squared",              False),
    "FABAR":           ("Mean Anomaly Forecast",         False),
    "OABAR":           ("Mean Anomaly Obs",              False),
    "FOABAR":          ("Mean Anomaly Forecast * Obs",   False),
    "FFABAR":          ("Mean Anomaly Forecast Squared", False),
    "OOABAR":          ("Mean Anomaly Obs Squared",      False),
}

KEY_COLUMNS_ONLY = False  # set True to keep only key columns

## Convert

Point `INPUT_FOLDER` at your GridStat output directory (containing `YYYYMMDDHH` subdirectories).  
The parquet file will be written to `OUTPUT_DIR`, named after the input folder.

In [3]:
INPUT_FOLDER = Path("input\AGlobal4_Analysis_T_AllLevels_00Z")  # change this
OUTPUT_DIR   = Path("output")                              # change this

convert(INPUT_FOLDER, OUTPUT_DIR)

parquet_path = OUTPUT_DIR / f"{INPUT_FOLDER.name}.parquet"

In [ ]:
# Apply key columns filter
if KEY_COLUMNS_ONLY:
    key_cols = [c for c, (_, is_key) in COLUMN_INFO.items() if is_key and c in df.columns]
    df = df[key_cols]

# Column reference table — key columns highlighted in bold
ref = pd.DataFrame(
    [(col, label, "★" if is_key else "") for col, (label, is_key) in COLUMN_INFO.items() if col in df.columns],
    columns=["Column", "Description", "Key"],
)

ref.style.apply(
    lambda row: ["font-weight: bold; background-color: #fffbe6"] * 3 if row["Key"] == "★" else [""] * 3,
    axis=1,
)

## Read the dataset

In [ ]:
df   = pq.read_table(parquet_path).to_pandas()
meta = {k.decode(): v.decode() for k, v in pq.read_table(parquet_path).schema.metadata.items()}

print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")
print(f"Model: {meta.get('MODEL')}  |  Variable: {meta.get('FCST_VAR')}")
df.head()

## Plot

Change `METRIC`, `X_AXIS`, and `GROUP_BY` to explore different views of the data.

## Exploring the dataset

Use the cells below to get familiar with the data before plotting.

In [ ]:
# Shape and column names
print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")
print(df.columns.tolist())

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Unique values in key columns — useful for knowing what to filter on
print("Forecast levels:  ", sorted(df["FCST_LEV"].unique()))
print("Domains (VX_MASK):", sorted(df["VX_MASK"].unique()))
print("Lead times (h):   ", sorted(df["FCST_LEAD_H"].unique()))

In [ ]:
# Filter example — change LEVEL and DOMAIN to suit your dataset
LEVEL  = "P850"       # change this
DOMAIN = "Australia"  # change this

df_filtered = df[(df["FCST_LEV"] == LEVEL) & (df["VX_MASK"] == DOMAIN)]
print(f"{len(df_filtered):,} rows after filtering")
df_filtered.head()

In [ ]:
METRIC   = "RMSE"         # column to plot on y-axis
X_AXIS   = "FCST_LEAD_H"  # column to plot on x-axis
GROUP_BY = "FCST_LEV"     # column to split into separate lines

fig, ax = plt.subplots(figsize=(10, 5))

for group, gdf in df.groupby(GROUP_BY):
    summary = gdf.groupby(X_AXIS)[METRIC].mean()
    ax.plot(summary.index, summary.values, marker="o", label=str(group))

ax.set_xlabel(X_AXIS)
ax.set_ylabel(METRIC)
ax.set_title(f"{METRIC} by {X_AXIS} — {meta.get('MODEL', '')} {meta.get('FCST_VAR', '')}")
ax.legend(title=GROUP_BY, bbox_to_anchor=(1, 1), loc="upper left")
plt.tight_layout()
plt.show()